# 第 1 集：训练 sin 模型

本 Notebook 只做一件事：**训练模型并导出产物**。
- 模型：3 层 MLP（1 -> 64 -> 64 -> 1，ReLU）
- 任务：拟合 `y = sin(x)`，其中 `x ∈ [0, 2π]`
- 产物：`sin_model.pth`、`sin_model.onnx`、`loss_history.csv`

> 提示：可视化请单独打开 `visualize.ipynb`，实现训练与可视化的职责分离。

## 1. 导入依赖

In [1]:
import csv
import os

import numpy as np
import onnx
import onnxsim
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, random_split

## 2. 参数配置

通过下方变量统一配置，学生可以直观看到每个参数的作用。

In [2]:
SEED = 42
EPOCHS = 500
NUM_SAMPLES = 1000
OUTPUT_DIR = "outputs"

torch.manual_seed(SEED)
os.makedirs(OUTPUT_DIR, exist_ok=True)
print("参数配置：")
print(f"  EPOCHS={EPOCHS}, NUM_SAMPLES={NUM_SAMPLES}, OUTPUT_DIR={OUTPUT_DIR}")

参数配置：
  EPOCHS=500, NUM_SAMPLES=1000, OUTPUT_DIR=outputs


## 3. 生成训练数据

在 `[0, 2π]` 区间生成带噪声的正弦样本，并划分 80% 训练 / 20% 验证。

In [3]:
def generate_data(num_samples=1000, noise=0.05):
    """生成 [0, 2π] 区间带噪声的正弦样本。"""
    x = torch.linspace(0, 2 * np.pi, num_samples)
    y = torch.sin(x) + noise * torch.randn(num_samples)
    return x.unsqueeze(1), y.unsqueeze(1)

x, y = generate_data(num_samples=NUM_SAMPLES)
dataset = TensorDataset(x, y)
n_val = NUM_SAMPLES // 5
n_train = NUM_SAMPLES - n_val
train_set, val_set = random_split(
    dataset, [n_train, n_val], generator=torch.Generator().manual_seed(SEED)
)
train_loader = DataLoader(train_set, batch_size=32, shuffle=True)
val_loader = DataLoader(val_set, batch_size=32, shuffle=False)
print(f"训练样本: {n_train}, 验证样本: {n_val}")

训练样本: 800, 验证样本: 200


## 4. 定义模型

In [4]:
class SinPredictor(nn.Module):
    def __init__(self):
        super(SinPredictor, self).__init__()
        self.fc1 = nn.Linear(1, 64)
        self.fc2 = nn.Linear(64, 64)
        self.fc3 = nn.Linear(64, 1)

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        x = self.fc3(x)
        return x

model = SinPredictor()
print(model)

SinPredictor(
  (fc1): Linear(in_features=1, out_features=64, bias=True)
  (fc2): Linear(in_features=64, out_features=64, bias=True)
  (fc3): Linear(in_features=64, out_features=1, bias=True)
)


## 5. 训练循环

每轮记录训练 loss 与验证 loss，方便后续可视化。

In [5]:
def evaluate(model, dataloader, criterion):
    model.eval()
    total = 0.0
    with torch.no_grad():
        for batch_x, batch_y in dataloader:
            total += criterion(model(batch_x), batch_y).item()
    return total / len(dataloader)

criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)

train_losses, val_losses = [], []
for epoch in range(EPOCHS):
    model.train()
    epoch_loss = 0.0
    for batch_x, batch_y in train_loader:
        y_pred = model(batch_x)
        loss = criterion(y_pred, batch_y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    train_loss = epoch_loss / len(train_loader)
    val_loss = evaluate(model, val_loader, criterion)
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    if (epoch + 1) % 50 == 0:
        print(f"Epoch [{epoch + 1}/{EPOCHS}], train loss: {train_loss:.5f}, val loss: {val_loss:.5f}")

print(f"final train loss: {train_losses[-1]:.5f}, final val loss: {val_losses[-1]:.5f}")

Epoch [50/500], train loss: 0.06680, val loss: 0.06580
Epoch [100/500], train loss: 0.02296, val loss: 0.02286
Epoch [150/500], train loss: 0.00944, val loss: 0.01170
Epoch [200/500], train loss: 0.00453, val loss: 0.00503
Epoch [250/500], train loss: 0.00316, val loss: 0.00382
Epoch [300/500], train loss: 0.00277, val loss: 0.00352
Epoch [350/500], train loss: 0.00263, val loss: 0.00316
Epoch [400/500], train loss: 0.00250, val loss: 0.00307
Epoch [450/500], train loss: 0.00266, val loss: 0.00326
Epoch [500/500], train loss: 0.00247, val loss: 0.00297
final train loss: 0.00247, final val loss: 0.00297


## 6. 保存训练产物

将 loss 历史写入 CSV，保存 PyTorch 权重，并导出 ONNX 模型。

In [6]:
# 1) loss 历史落盘
csv_path = os.path.join(OUTPUT_DIR, "loss_history.csv")
with open(csv_path, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["epoch", "train_loss", "val_loss"])
    for i, (tl, vl) in enumerate(zip(train_losses, val_losses), start=1):
        writer.writerow([i, f"{tl:.6f}", f"{vl:.6f}"])
print(f"saved: {csv_path}")

# 2) 保存 PyTorch 权重
pth_path = os.path.join(OUTPUT_DIR, "sin_model.pth")
torch.save(model.state_dict(), pth_path)
print(f"saved: {pth_path}")

# 3) 导出 ONNX
onnx_path = os.path.join(OUTPUT_DIR, "sin_model.onnx")
dummy_input = torch.randn([1, 1], dtype=torch.float32)
torch.onnx.export(
    model,
    dummy_input,
    onnx_path,
    opset_version=12,
    input_names=["input"],
    output_names=["output"],
    dynamo=False,
)
onnx_model = onnx.load_model(onnx_path)
onnx.checker.check_model(onnx_model)
onnx_model, check = onnxsim.simplify(onnx_model)
assert check, "Simplified ONNX model could not be validated"
onnx.save(onnx_model, onnx_path)
print(f"saved: {onnx_path}")

saved: outputs/loss_history.csv
saved: outputs/sin_model.pth
saved: outputs/sin_model.onnx


/var/folders/3r/dy1krlx54vs0m2jmmbsd15lc0000gn/T/ipykernel_43723/2379255210.py:18: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(
